# 第 4 週 實作｜導數的應用 II(Colab notebook)

**目標**:把「找極值」從手算搬到電腦上,親手體會最佳化的數值版——
1. `gradient_descent(df, x0, lr, steps)`:沿 $-f'$ 一步步走下山找最小值(在 $x^2$ 與一個多谷函數上畫軌跡,示範會**卡在局部最小**)
2. 線性回歸:對雜訊資料用梯度下降最小化 **MSE**,把直線擬合出來(最佳化的數值版)
3. SymPy **自動曲線描繪**:給 $f$ 自動求 $f'$、$f''$,解臨界點與反曲點並標在圖上

**用法**:上傳到 [Google Colab](https://colab.research.google.com/) 或本機 Jupyter,由上往下逐格執行。
標「`# TODO 學生練習`」的格子留給學生填。

> 圖表標籤用英文/數學符號以避免中文變豆腐字;中文都放在說明格。

In [ ]:
# === 環境設定(先跑這格)===
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['axes.grid'] = True
print("環境就緒, numpy", np.__version__)

### (選用)讓圖表顯示中文

預設圖表用英文標籤,避免中文變「豆腐字」。若你在 Colab 想要中文座標/標題,
把下一格的註解取消再執行(只需一次),之後的圖就能顯示中文。

In [ ]:
# 想要中文圖標時,取消以下註解執行(Colab 適用;本機 Jupyter 需自備 CJK 字型)
# !apt-get -qq install fonts-noto-cjk > /dev/null
# import matplotlib
# matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# matplotlib.rcParams['axes.unicode_minus'] = False
print("預設英文標籤;要中文請見上一格說明")

## Lab 1｜梯度下降 gradient descent

手算最佳化是解 $f'(x)=0$;電腦不解方程,而是**沿著下坡方向一步步走**:
$$x_{\text{new}} = x_{\text{old}} - \text{lr}\cdot f'(x_{\text{old}}).$$
`lr`(learning rate)是步伐大小。先在碗形 $f(x)=x^2$ 上看它乖乖滾到谷底。

In [ ]:
def gradient_descent(df, x0, lr, steps):
    """沿 -df 方向走下山,回傳每一步的 x(軌跡)。df 是導函數 f'。"""
    xs = [float(x0)]
    x = float(x0)
    for _ in range(steps):
        x = x - lr * df(x)      # 往下坡走一步
        xs.append(x)
    return xs

# 碗形 f(x)=x^2, f'(x)=2x
f  = lambda x: x**2
df = lambda x: 2*x
path = gradient_descent(df, x0=5.0, lr=0.1, steps=30)
print("從 x0=5 出發,最後停在 x =", round(path[-1], 6), " (理論最小值在 x=0)")

xx = np.linspace(-5.5, 5.5, 400)
plt.plot(xx, f(xx), label='f(x) = x^2')
px = np.array(path)
plt.plot(px, f(px), 'o-', color='C3', ms=4, label='descent path')
plt.title('Gradient descent on a convex bowl -> global min'); plt.legend(); plt.show()

### 卡在局部最小(local minimum)

換一個**多谷**函數 $f(x)=3x^4-16x^3+18x^2$。它有一個**淺谷**(局部最小,$x=0$)和一個**深谷**(全域最小,$x=3$)。
梯度下降只會往「腳邊的下坡」走——**起點不同,可能掉進不同的谷**。這正是深度學習最惱人的問題之一。

In [ ]:
g  = lambda x: 3*x**4 - 16*x**3 + 18*x**2
dg = lambda x: 12*x**3 - 48*x**2 + 36*x      # g'(x)

pathA = gradient_descent(dg, x0=0.5, lr=0.01, steps=40)   # 起點靠近淺谷
pathB = gradient_descent(dg, x0=2.0, lr=0.01, steps=40)   # 起點靠近深谷
print("從 x0=0.5 出發 -> x =", round(pathA[-1], 4), " f =", round(g(pathA[-1]), 3), " (局部最小)")
print("從 x0=2.0 出發 -> x =", round(pathB[-1], 4), " f =", round(g(pathB[-1]), 3), " (全域最小)")

xx = np.linspace(-0.7, 4.2, 500)
plt.plot(xx, g(xx), 'k', label='f(x)=3x^4-16x^3+18x^2')
for p, c, lab in [(pathA, 'C0', 'start 0.5 -> local min'), (pathB, 'C3', 'start 2.0 -> global min')]:
    p = np.array(p); plt.plot(p, g(p), 'o-', color=c, ms=3, label=lab)
plt.title('Same rule, different start -> may get stuck in a local minimum')
plt.legend(fontsize=8); plt.ylim(-30, 40); plt.show()

In [ ]:
# TODO 學生練習:
# (1) 把 lr 從 0.01 慢慢調大(0.02, 0.03, ...),觀察步伐變大後會發生什麼(可能震盪甚至發散)。
# (2) 從 x0=0.9 出發(靠近局部最大 x=1),它最後掉進哪個谷?為什麼?
# path = gradient_descent(dg, x0=0.9, lr=0.01, steps=40)
# print(round(path[-1], 4))

## Lab 2｜用梯度下降做線性回歸(最佳化的實戰版)

給一堆有雜訊的點,想找一條直線 $y=wx+b$ 最貼合它們。「最貼合」= 讓**均方誤差(MSE)**最小:
$$\text{MSE}(w,b)=\frac1N\sum_i (w x_i + b - y_i)^2.$$
這是一個「對 $w,b$ 的最佳化問題」。我們對 $w,b$ 各求偏導,再用**同一套梯度下降**把 MSE 一路壓低。

In [ ]:
# 造一組雜訊資料:真線是 y = 2x + 1
rng = np.random.default_rng(0)
N = 60
xdat = np.linspace(0, 5, N)
ydat = 2.0*xdat + 1.0 + rng.normal(0, 1.2, N)

def mse(w, b):
    return np.mean((w*xdat + b - ydat)**2)

# 對 MSE 做梯度下降:d/dw = 2*mean(err*x), d/db = 2*mean(err)
w, b, lr = 0.0, 0.0, 0.02
loss_hist = []
for _ in range(400):
    err = (w*xdat + b) - ydat
    w -= lr * 2*np.mean(err*xdat)
    b -= lr * 2*np.mean(err)
    loss_hist.append(mse(w, b))

print("擬合結果 w =", round(w,3), "(真值 2),  b =", round(b,3), "(真值 1),  最終 MSE =", round(loss_hist[-1],3))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(loss_hist); ax[0].set_title('MSE loss vs iteration'); ax[0].set_xlabel('step'); ax[0].set_ylabel('MSE')
ax[1].scatter(xdat, ydat, s=14, label='noisy data')
ax[1].plot(xdat, w*xdat + b, 'C3', label='GD fit: y=%.2fx+%.2f' % (w, b))
ax[1].set_title('Line fitted by gradient descent'); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# TODO 學生練習:
# (1) 把 lr 改成 0.001,跑同樣 400 步,loss 降得夠不夠?再改 0.05 看會不會震盪。
# (2) 把資料的雜訊 std 從 1.2 加大到 3.0,擬合的 w,b 還準嗎?
# 這就是機器學習訓練的雛形:定義 loss -> 求梯度 -> 沿梯度下降。

## Lab 3｜SymPy 自動曲線描繪

理論課的曲線描繪流程(求 $f'$、$f''$、解臨界點與反曲點)可以完全自動化。
下面的 `auto_sketch` 用 SymPy 幫你**算導數、解方程**,再把臨界點(●)與反曲點(■)標到圖上。

In [ ]:
x = sp.symbols('x')

def auto_sketch(expr, xmin=-1.5, xmax=3.5):
    f1 = sp.diff(expr, x)
    f2 = sp.diff(expr, x, 2)
    crit = [c for c in sp.solve(f1, x) if c.is_real]
    # 反曲點:f''=0 且該處 f''' != 0(確保 f'' 真的變號)
    infl = [c for c in sp.solve(f2, x) if c.is_real and sp.diff(expr, x, 3).subs(x, c) != 0]
    print("f   =", expr)
    print("f'  =", sp.factor(f1), "  critical points:", crit)
    print("f'' =", sp.factor(f2), "  inflection points:", infl)

    fpy = sp.lambdify(x, expr, "numpy")
    xs = np.linspace(xmin, xmax, 500)
    plt.plot(xs, fpy(xs), 'k', label='f(x)')
    for c in crit:
        plt.plot(float(c), float(expr.subs(x, c)), 'o', color='C3', ms=9,
                 label='critical' if c == crit[0] else None)
    for c in infl:
        plt.plot(float(c), float(expr.subs(x, c)), 's', color='C0', ms=9,
                 label='inflection' if c == infl[0] else None)
    plt.title('Automatic curve sketching (SymPy)'); plt.legend(); plt.show()
    return crit, infl

auto_sketch(x**3 - 3*x**2 + 2, -1.5, 3.5)

In [ ]:
# TODO 學生練習:換一個函數自動描繪,並和你手算的結果核對
# auto_sketch(x**4 - 6*x**2, -3, 3)          # 期望 critical: 0, +-sqrt(3);inflection: +-1
# auto_sketch(3*x**4 - 4*x**3, -0.8, 1.6)    # 期望 critical: 0(非極值), 1;inflection: 0, 2/3

## 收尾 · 與筆試 / CS 的連結

| 這格實作 | 對應觀念 | 通向 |
|---|---|---|
| Lab 1 gradient descent | 極值、最佳化(觀念 1、10) | 深度學習的訓練引擎 |
| Lab 1 多谷函數 | 局部 vs 全域極值(觀念 3、4) | local minimum 難題 |
| Lab 2 線性回歸 (MSE) | 最佳化的數值版(觀念 10) | 機器學習 model fitting |
| Lab 3 SymPy 自動描繪 | 曲線描繪、$f'$/$f''$(觀念 5、9) | 符號運算 / CAS |

### 進階徽章(選做)
1. 讓 `gradient_descent` **回傳每步的 $|f'(x)|$**,畫出它如何趨近 $0$(收斂診斷)。
2. 幫 Lab 2 加上一個「動量(momentum)」項,看 loss 是否降得更快。
3. 用 `sp.solve(sp.diff(expr, x), x)` 驗證你手算最佳化題(圍籬、罐頭)的臨界點。